## Step 5 — Prompt Creation for Metadata and Story

In addition to the geometric reconstruction pipeline, each object in the dataset
includes a textual description generated through a structured prompting process.

This step captures the **artistic, cultural, and emotional context** of each
kintsugi or yobitsugi piece—information that cannot be expressed through 3D shape
or color alone.

The prompt consists of two main components:

---

### A. Metadata (Objective Information)

Structured factual attributes describing the ceramic object:

- **Material / texture** of the plate
- **Color of the kintsugi lines**
- **Brand or kiln**, place of origin, region
- **Date or era** when it was produced
- **Intentionality of the break** (accidental, natural aging, etc.)
- **Restoration artist / author** (individual, workshop, or anonymous)

This information helps ground the object within its historical and cultural
background.

---

### B. Story (Subjective Information)

Narrative elements conveying the emotion and meaning behind the repair:

- The **reason** the piece was repaired
- The **workshop or environment** where the repair was performed
- The **values or philosophy** guiding the kintsugi
- How the **creator felt** during the restoration process

This part reflects the human-centered aspect of kintsugi and allows the dataset
to include cultural richness beyond geometric reconstruction.

---

### Example Prompt Structure

In [ ]:
import os
import json
from google import genai
from google.genai import types

# ==========
# 1. API KEY
# ==========
os.environ["GOOGLE_API_KEY"] = "API_KEY"

client = genai.Client()

# ==========
# 2. JSON TEMPLATE
# ==========
json_template = {
    "metadata": {
        "owner": {
            "plate_material": "",
            "kintsugi_line_color": "",
            "plate_brand": "",
            "date_of_breakage": ""
        },
        "kintsugi_artisan": {
            "date_created": "",
            "artisan": ""
        }
    },
    "story": {
        "owner": {
            "motivation_for_kintsugi": "",
            "feelings_after_repair": ""
        },
        "kintsugi_artisan": {
            "techniques_and_considerations": "",
            "concept_behind_the_work": ""
        }
    }
}

# ==========
# 3. TEXT INPUT
# ==========
text_input = """text"""

with open("step1.png", "rb") as f:
    image_bytes = f.read()

image_part = types.Part.from_bytes(
    data=image_bytes,
    mime_type="image/png"
)

prompt = f"""
Fill the JSON template based on the given TEXT and IMAGE.

Rules:
- Follow the JSON structure EXACTLY
- Fill fields using TEXT first
- If not available, infer from IMAGE
- Actively infer visual metadata (material, color, brand style)
- Do NOT leave metadata empty if it can be visually estimated
- Extract implicit meanings when clearly suggested
- Do not leave story fields empty if emotional intent exists
- If still unknown, return ""
- Output ONLY JSON

JSON Template:
{json.dumps(json_template)}

TEXT:
{text_input}
"""
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[prompt, image_part],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        temperature=0.0
    ),
)
print("===== RAW =====")
print(response.text)
try:
    output_json = json.loads(response.text)

    print("\n===== PARSED =====")
    print(json.dumps(output_json, indent=2, ensure_ascii=False))
    with open("output.json", "w", encoding="utf-8") as f:
        json.dump(output_json, f, indent=2, ensure_ascii=False)

    print("Saved to output.json")

except Exception as e:
    print("JSON parse failed:", e)